# Modified Swiss Dwellings: Floor Plan to Graph Conversion

This notebook demonstrates converting floor plan data into topological graphs using topologic_fast.

**Adapted from topologicpy MSD notebook**

The Modified Swiss Dwellings is a machine learning-ready floor plan dataset created at Delft University.
Dataset license: CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/)

## Key Concepts

1. Representing floor plans as graphs
2. Rooms as vertices, adjacencies as edges
3. Storing semantic attributes (room types, zones)
4. Visualization with Plotly

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np

## Room Type Definitions

Define colors and semantic types for different room categories.

In [ ]:
# Room type colors (matching Swiss Dwellings dataset)
ROOM_COLORS = {
    'Bedroom': '#1f77b4',
    'Livingroom': '#e6550d',
    'Kitchen': '#fd8d3c',
    'Dining': '#fdae6b',
    'Corridor': '#fdd0a2',
    'Stairs': '#72246c',
    'Storeroom': '#5254a3',
    'Bathroom': '#6b6ecf',
    'Balcony': '#2ca02c',
}

# Zone groupings
ZONE_GROUPS = {
    'Zone1': ['Bedroom'],
    'Zone2': ['Livingroom', 'Kitchen', 'Dining', 'Corridor'],
    'Zone3': ['Stairs', 'Storeroom', 'Bathroom'],
    'Zone4': ['Balcony']
}

ZONE_COLORS = {
    'Zone1': '#1f77b4',   # Blue - Private/Sleep
    'Zone2': '#ff7f0e',   # Orange - Living
    'Zone3': '#72246c',   # Purple - Service
    'Zone4': '#2ca02c',   # Green - Outdoor
}

print("Room types:", list(ROOM_COLORS.keys()))
print("Zone groups:", list(ZONE_GROUPS.keys()))

## Create Sample Floor Plan

Since we don't have the actual MSD dataset, we create a sample floor plan
representing a typical apartment layout.

```
+----------+--------+----------+
|          |        |          |
| Bedroom1 | Bath   | Bedroom2 |
|          |        |          |
+----+-----+--------+-----+----+
|    |                    |    |
|Stor|      Corridor      |Strs|
|    |                    |    |
+----+--------+-----------+----+
|             |                |
|  Kitchen    |   Livingroom   |
|             |                |
+-----+-------+-------+--------+
|     |               |        |
|Dinng|   Livingroom  | Balcny |
|     |               |        |
+-----+---------------+--------+
```

In [ ]:
# Define rooms with their geometry and type
# Format: (name, type, x, y, width, height)
rooms_data = [
    ('Bedroom1', 'Bedroom', 0, 9, 4, 3),
    ('Bathroom', 'Bathroom', 4, 9, 3, 3),
    ('Bedroom2', 'Bedroom', 7, 9, 4, 3),
    ('Storage', 'Storeroom', 0, 6, 1.5, 3),
    ('Corridor', 'Corridor', 1.5, 6, 7.5, 3),
    ('Stairs', 'Stairs', 9, 6, 2, 3),
    ('Kitchen', 'Kitchen', 0, 3, 5, 3),
    ('Living1', 'Livingroom', 5, 3, 6, 3),
    ('Dining', 'Dining', 0, 0, 2.5, 3),
    ('Living2', 'Livingroom', 2.5, 0, 6.5, 3),
    ('Balcony', 'Balcony', 9, 0, 2, 3),
]

# Create room geometries as Faces
rooms = []
room_info = []

for name, rtype, x, y, w, h in rooms_data:
    # Create rectangular face for the room
    face = tf.Face.Rectangle(width=w, length=h)
    
    # Calculate centroid position for translation
    cx = x + w/2
    cy = y + h/2
    
    # NOTE: Topology.Translate not yet available, store transform info
    area = w * h
    
    rooms.append(face)
    room_info.append({
        'name': name,
        'type': rtype,
        'x': x,
        'y': y,
        'width': w,
        'height': h,
        'centroid': (cx, cy),
        'area': area,
        'color': ROOM_COLORS.get(rtype, '#999999')
    })

print(f"Created {len(rooms)} rooms:")
for info in room_info:
    print(f"  {info['name']:12s} [{info['type']:10s}] - {info['area']:.1f} m^2")

total_area = sum(info['area'] for info in room_info)
print(f"\nTotal floor area: {total_area:.1f} m^2")

## Define Room Adjacencies

Define which rooms are adjacent to each other (share a wall or opening).

In [ ]:
# Define adjacencies (room index pairs that share a boundary)
adjacencies = [
    # Row 1: Bedrooms and Bathroom
    ('Bedroom1', 'Bathroom'),
    ('Bathroom', 'Bedroom2'),
    
    # Row 1 to Row 2: Connection to corridor
    ('Bedroom1', 'Storage'),
    ('Bedroom1', 'Corridor'),
    ('Bathroom', 'Corridor'),
    ('Bedroom2', 'Corridor'),
    ('Bedroom2', 'Stairs'),
    
    # Row 2: Storage, Corridor, Stairs
    ('Storage', 'Corridor'),
    ('Corridor', 'Stairs'),
    
    # Row 2 to Row 3: Connection to Kitchen/Living
    ('Storage', 'Kitchen'),
    ('Corridor', 'Kitchen'),
    ('Corridor', 'Living1'),
    ('Stairs', 'Living1'),
    
    # Row 3: Kitchen and Living
    ('Kitchen', 'Living1'),
    
    # Row 3 to Row 4
    ('Kitchen', 'Dining'),
    ('Living1', 'Living2'),
    ('Living1', 'Balcony'),
    
    # Row 4: Dining, Living, Balcony
    ('Dining', 'Living2'),
    ('Living2', 'Balcony'),
]

print(f"Defined {len(adjacencies)} adjacency relationships")

## Create Floor Plan Graph

Build a graph where:
- Vertices represent room centroids
- Edges represent adjacencies between rooms

In [ ]:
# Create vertices at room centroids
vertices = {}
vertex_list = []

for info in room_info:
    cx, cy = info['centroid']
    v = tf.Vertex.ByCoordinates(cx, cy, 0)
    vertices[info['name']] = v
    vertex_list.append(v)

# Create edges from adjacencies
edges = []
for room1, room2 in adjacencies:
    if room1 in vertices and room2 in vertices:
        edge = tf.Edge.ByStartVertexEndVertex(vertices[room1], vertices[room2])
        edges.append(edge)

# Create graph
graph = tf.Graph.ByVerticesEdges(vertex_list, edges)

print(f"Floor Plan Graph:")
print(f"  Rooms (vertices): {graph.Order()}")
print(f"  Adjacencies (edges): {graph.Size()}")
print(f"  Density: {graph.Density():.3f}")
print(f"  Diameter: {graph.Diameter()} steps")

## Room Connectivity Analysis

In [ ]:
# Analyze connectivity for each room
print("Room Connectivity:")
print("=" * 60)

# Create lookup for vertex to room info
def get_room_by_vertex(v):
    coords = v.Coordinates()
    for info in room_info:
        cx, cy = info['centroid']
        if abs(coords[0] - cx) < 0.1 and abs(coords[1] - cy) < 0.1:
            return info
    return None

connectivity_data = []

for info in room_info:
    v = vertices[info['name']]
    degree = graph.VertexDegree(v)
    adjacent = graph.AdjacentVertices(v)
    adj_names = []
    for adj_v in adjacent:
        adj_info = get_room_by_vertex(adj_v)
        if adj_info:
            adj_names.append(adj_info['name'])
    
    connectivity_data.append({
        'name': info['name'],
        'type': info['type'],
        'degree': degree,
        'neighbors': adj_names
    })

# Sort by connectivity
connectivity_data.sort(key=lambda x: -x['degree'])

for data in connectivity_data:
    print(f"\n{data['name']} [{data['type']}]: {data['degree']} connections")
    for neighbor in sorted(data['neighbors']):
        print(f"    -> {neighbor}")

## Visualize Floor Plan with Graph

In [ ]:
def visualize_floorplan_graph(room_info, graph):
    """Create 2D visualization of floor plan with connectivity graph."""
    fig = go.Figure()
    
    # Draw rooms as rectangles
    for info in room_info:
        x = info['x']
        y = info['y']
        w = info['width']
        h = info['height']
        
        # Create rectangle corners
        rect_x = [x, x+w, x+w, x, x]
        rect_y = [y, y, y+h, y+h, y]
        
        fig.add_trace(go.Scatter(
            x=rect_x, y=rect_y,
            fill='toself',
            fillcolor=info['color'],
            line=dict(color='black', width=2),
            name=f"{info['name']} ({info['type']})",
            hoverinfo='name',
            opacity=0.7
        ))
    
    # Draw graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='rgba(0,0,0,0.6)', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw graph vertices with labels
    for info in room_info:
        cx, cy = info['centroid']
        fig.add_trace(go.Scatter(
            x=[cx], y=[cy],
            mode='markers+text',
            marker=dict(
                size=15,
                color='white',
                line=dict(color='black', width=2)
            ),
            text=[info['name'].replace('Living', 'Liv').replace('Bedroom', 'Bed')],
            textposition='middle center',
            textfont=dict(size=8),
            showlegend=False,
            hovertext=f"{info['name']}: {info['area']:.1f} m^2",
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title='Floor Plan with Room Adjacency Graph',
        xaxis=dict(
            title='X (m)',
            scaleanchor='y',
            scaleratio=1,
            range=[-1, 12]
        ),
        yaxis=dict(
            title='Y (m)',
            range=[-1, 13]
        ),
        width=800,
        height=700,
        showlegend=True,
        legend=dict(x=1.02, y=1, font=dict(size=9))
    )
    
    return fig

fig = visualize_floorplan_graph(room_info, graph)
fig.show()

## Zone Analysis

Group rooms by functional zones and analyze zone connectivity.

In [ ]:
# Assign zones to rooms
def get_zone(room_type):
    for zone, types in ZONE_GROUPS.items():
        if room_type in types:
            return zone
    return 'Unknown'

# Add zone info
for info in room_info:
    info['zone'] = get_zone(info['type'])
    info['zone_color'] = ZONE_COLORS.get(info['zone'], '#999999')

# Summarize zones
zone_stats = {}
for info in room_info:
    zone = info['zone']
    if zone not in zone_stats:
        zone_stats[zone] = {'count': 0, 'area': 0, 'rooms': []}
    zone_stats[zone]['count'] += 1
    zone_stats[zone]['area'] += info['area']
    zone_stats[zone]['rooms'].append(info['name'])

print("Zone Summary:")
print("=" * 50)
for zone in sorted(zone_stats.keys()):
    stats = zone_stats[zone]
    print(f"\n{zone}:")
    print(f"  Rooms: {stats['count']}")
    print(f"  Area: {stats['area']:.1f} m^2")
    print(f"  Members: {', '.join(stats['rooms'])}")

In [ ]:
def visualize_by_zone(room_info, graph):
    """Visualize floor plan colored by zone."""
    fig = go.Figure()
    
    # Draw rooms colored by zone
    for info in room_info:
        x = info['x']
        y = info['y']
        w = info['width']
        h = info['height']
        
        rect_x = [x, x+w, x+w, x, x]
        rect_y = [y, y, y+h, y+h, y]
        
        fig.add_trace(go.Scatter(
            x=rect_x, y=rect_y,
            fill='toself',
            fillcolor=info['zone_color'],
            line=dict(color='black', width=2),
            name=f"{info['name']} ({info['zone']})",
            hoverinfo='name',
            opacity=0.7
        ))
    
    # Draw graph
    graph_edges = graph.Edges()
    for edge in graph_edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='black', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    fig.update_layout(
        title='Floor Plan by Functional Zone',
        xaxis=dict(title='X (m)', scaleanchor='y', scaleratio=1, range=[-1, 12]),
        yaxis=dict(title='Y (m)', range=[-1, 13]),
        width=800,
        height=700
    )
    
    return fig

fig_zones = visualize_by_zone(room_info, graph)
fig_zones.show()

## Path Finding Analysis

In [ ]:
# Find paths between key rooms
path_queries = [
    ('Bedroom1', 'Kitchen'),
    ('Bedroom2', 'Balcony'),
    ('Bathroom', 'Living2'),
    ('Storage', 'Balcony'),
]

print("Path Analysis:")
print("=" * 50)

for start_name, end_name in path_queries:
    start_v = vertices[start_name]
    end_v = vertices[end_name]
    
    distance = graph.Distance(start_v, end_v)
    path = graph.Path(start_v, end_v)
    
    if path:
        path_vertices = path.Vertices()
        path_names = []
        for pv in path_vertices:
            pinfo = get_room_by_vertex(pv)
            if pinfo:
                path_names.append(pinfo['name'])
        
        print(f"\n{start_name} -> {end_name}:")
        print(f"  Distance: {distance} rooms")
        print(f"  Path: {' -> '.join(path_names)}")
    else:
        print(f"\n{start_name} -> {end_name}: No path found")

## Area Distribution by Room Type

In [ ]:
# Aggregate by room type
type_areas = {}
for info in room_info:
    rtype = info['type']
    if rtype not in type_areas:
        type_areas[rtype] = 0
    type_areas[rtype] += info['area']

# Create pie chart
fig_pie = go.Figure(data=[go.Pie(
    labels=list(type_areas.keys()),
    values=list(type_areas.values()),
    hole=0.3,
    marker_colors=[ROOM_COLORS.get(t, '#999999') for t in type_areas.keys()]
)])

fig_pie.update_layout(
    title='Floor Area by Room Type',
    width=600,
    height=500
)

fig_pie.show()

## Summary

This notebook demonstrated:

1. **Floor Plan Representation**: Rooms as geometric elements with attributes
2. **Graph Construction**: Creating connectivity graphs from adjacency relationships
3. **Connectivity Analysis**: Examining room relationships and pathfinding
4. **Zone Grouping**: Organizing rooms by functional zones
5. **Visualization**: Multiple views of the floor plan data

### Not Yet Implemented in topologic_fast:
- `Topology.SetDictionary()` / `Dictionary` class - Storing attributes on topologies
- `Topology.Translate()` - Moving topologies in space
- `Graph.ByNetworkXGraph()` - Import from NetworkX
- `Graph.ExportToJSON()` - Export to JSON format

### Applications:
- Space syntax analysis
- Wayfinding optimization
- Floor plan ML datasets
- Building performance simulation